# Bose2 — Kaggle free-GPU trial: cinematic crochet + Egyptian Arabic voice

Manual experiment only. In Kaggle Notebook Settings, choose GPU if available at no cost. Enable Internet if available for model and voice downloads. If GPU is unavailable or quota is exhausted, stop; do not add billing or paid services. This notebook does not upload or publish anything.

In [ ]:
%pip -q install -U diffusers transformers accelerate imageio-ffmpeg edge-tts
import torch, os
if not torch.cuda.is_available():
    raise RuntimeError('Kaggle GPU is not enabled/available. Open Notebook Settings > Accelerator > GPU if available for free; do not pay.')
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))
import imageio_ffmpeg
FFMPEG = imageio_ffmpeg.get_ffmpeg_exe()
print('FFmpeg:', FFMPEG)

In [ ]:
from pathlib import Path
import subprocess, torch
from diffusers import CogVideoXPipeline
from diffusers.utils import export_to_video
from IPython.display import Video, display, FileLink
out = Path('/kaggle/working/bose2_trial'); out.mkdir(parents=True, exist_ok=True)
prompt = 'Cinematic macro shot of hands crocheting cream yarn with a crochet hook on a wooden table, warm natural window light, shallow depth of field, slow camera movement, realistic yarn texture, documentary craft film, stable framing, no text.'
script = 'النهارده هنتعلم بداية غرزة كروشيه بسيطة خطوة بخطوة.'
pipe = CogVideoXPipeline.from_pretrained('zai-org/CogVideoX-2b', torch_dtype=torch.float16)
pipe.enable_model_cpu_offload(); pipe.vae.enable_slicing(); pipe.vae.enable_tiling()
frames = pipe(prompt=prompt, num_videos_per_prompt=1, num_inference_steps=30, num_frames=49, guidance_scale=6, generator=torch.Generator(device='cuda').manual_seed(42)).frames[0]
silent = out / 'scene.mp4'; export_to_video(frames, str(silent), fps=8)
if not silent.exists() or silent.stat().st_size < 1024: raise RuntimeError('No valid video was generated.')
audio = out / 'voice.mp3'; final = out / 'bose2_ai_voice_video_test.mp4'
subprocess.run(['edge-tts','--voice','ar-EG-SalmaNeural','--rate=-5%','--text',script,'--write-media',str(audio)], check=True, timeout=120)
subprocess.run([FFMPEG,'-y','-i',str(silent),'-i',str(audio),'-map','0:v:0','-map','1:a:0','-c:v','copy','-c:a','aac','-shortest','-movflags','+faststart',str(final)], check=True, timeout=180)
if not final.exists() or final.stat().st_size < 1024: raise RuntimeError('Final video invalid.')
print('Generated:', final, 'bytes:', final.stat().st_size)
display(Video(str(final), embed=True))
display(FileLink(str(final)))

## Review gate — do not skip

- Play the MP4 and listen to the Egyptian Arabic voice.
- Check whether the generated hands, hook, yarn and stitch motion are plausible; AI video may be visually wrong or flicker.
- This is a short visual/audio test, not a complete 1–3 minute lesson. Arabic captions are not burned into this sample yet.
- Download the MP4 from the link above. Do not upload it publicly. Bose2 production remains unchanged until a person reviews and approves the clip.